## PREPROCESAMIENTO DEL DATASET LAPOP BOLIVIA 2023 POR PASOS

PASO 1 - Carga del dataset (loader.py)

In [8]:
import sys
from pathlib import Path
project_root = Path.cwd().parent.parent 
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.preprocessing.loader import DatasetLoader
from src.utils.constants import RAW_DATA_PATH

print("=" * 70)
print("BLOQUE 1: TEST DATASETLOADER")
print("=" * 70)

dataset_path = RAW_DATA_PATH
loader = DatasetLoader(dataset_path)
df = loader.load()

print(f"\n✓ Dataset cargado exitosamente")
print(f"  Dimensiones: {df.shape[0]} registros × {df.shape[1]} variables")
print(f"\nPrimeras 5 filas:")
print(df.head())

print(f"\nVariables seleccionadas:")
for col in df.columns:
    print(f"  - {col}")

print(f"\nMissings por variable:")
missing_counts = df.isnull().sum()
for var, count in missing_counts.items():
    if count > 0:
        pct = (count / len(df)) * 100
        print(f"  {var:15} {count:4} ({pct:5.1f}%)")

print("\n" + "=" * 70)

BLOQUE 1: TEST DATASETLOADER
Cargando dataset desde: C:\electocluster\data\raw\lapop_bolivia_2023.dta
✓ Dataset cargado: 1706 registros × 208 variables

✓ Dataset cargado exitosamente
  Dimensiones: 1706 registros × 208 variables

Primeras 5 filas:
   idnum  pais  nationality  estratopri  estratosec  strata   prov  municipio  \
0   5999    10           10        1001           1    1001  10201    1020101   
1   3621    10           10        1001           1    1001  10201    1020101   
2   5202    10           10        1001           3    1001  10206    1020602   
3    980    10           10        1001           1    1001  10201    1020101   
4   5992    10           10        1001           3    1001  10220    1022001   

   upm  ur  ...  r16  r27  colorr  noise1  conocim  sexin  colori      fecha  \
0   82   1  ...  1.0  0.0       3       0        1      2       4 2023-05-03   
1   56   1  ...  1.0  0.0       6       1        3      2       4 2023-04-22   
2   20   1  ...  1.0  1.

PASO 2 - Limpieza del dataset (cleaner.py)

In [9]:
from src.preprocessing.cleaner import DataCleaner

print("=" * 70)
print("BLOQUE 2: TEST DATACLEANER")
print("=" * 70)

# Usar el df del BLOQUE 1
print(f"\nDataset antes de limpieza:")
print(f"  Dimensiones: {df.shape}")
print(f"  Missings totales: {df.isnull().sum().sum()}")

cleaner = DataCleaner(df.copy())

# Ejecutar limpieza paso a paso
print("\n--- Validando rangos ---")
cleaner.validate_ranges()

print("\n--- Imputando valores faltantes ---")
cleaner.handle_missing_values()

print("\n--- Detectando outliers ---")
outliers = cleaner.detect_outliers(method='iqr', threshold=1.5)

# Obtener dataset limpio
df_clean = df

print(f"\nDataset después de limpieza:")
print(f"  Dimensiones: {df_clean.shape}")
print(f"  Missings totales: {df_clean.isnull().sum().sum()}")

# Verificar cuántos registros tienen boletidnew=0 (No indígena)
# pero boletidnewb con valor asignado (no NaN)
mascara = (df['boletidnew'] == 0) & (df['boletidnewb'] == 0)
print(mascara.sum())
# Si esto devuelve > 0, la imputación contaminó los datos

print(f"\nPrimeras 5 filas del dataset limpio:")
print(df_clean.head())

print("\n" + "=" * 70)

BLOQUE 2: TEST DATACLEANER

Dataset antes de limpieza:
  Dimensiones: (1706, 208)
  Missings totales: 55073

--- Validando rangos ---
Validando rangos de variables...
✓ Validación de rangos completada

--- Imputando valores faltantes ---
Imputando valores faltantes...
  ✓ edre: 5 missings imputados con moda (4.0)
  ✓ q10inc: 199 missings imputados con moda (1003.0)
  ✓ etid: 189 missings imputados con moda (2.0)
  ✓ boletidnew: 54 missings imputados con moda (1.0)
  ✓ boletidnewb: 850 missings imputados con moda (1.0)
  ✓ ocupoit: 740 missings imputados con moda (5.0)
  ✓ q1tc_r: 58 missings imputados con moda (1.0)
  ✓ q3cn: 62 missings imputados con moda (1.0)
  ✓ q5b: 16 missings imputados con moda (1.0)
  ✓ r3: 12 missings imputados con moda (1.0)
  ✓ r4a: 7 missings imputados con moda (1.0)
  ✓ r6: 8 missings imputados con moda (0.0)
  ✓ r7: 6 missings imputados con moda (0.0)
  ✓ r12: 6 missings imputados con moda (1.0)
  ✓ r15: 8 missings imputados con moda (0.0)
  ✓ r18n: 11 mi

PASO 3 - Transformación de datos del dataset (transformer.py)

In [10]:

from src.preprocessing.transformer import DataTransformer
from src.preprocessing.feature_engineering import engineer_features

print("=" * 70)
print("BLOQUE 3: TEST DATATRANSFORMER")
print("=" * 70)

# Usar el df_clean del BLOQUE 2
print(f"\nDataset antes de transformación:")
print(f"  Dimensiones: {df_clean.shape}")
print(df_clean.dtypes)

# Feature engineering previo a la transformación
print(f"\n--- Feature Engineering ---")
df_engineered, _, engineering_report = engineer_features(
    df_clean.copy(),
    build_wealth=True,
    wealth_method='sum',
    build_civic=True,
    drop_originals=True
)
print(f"  Dimensiones post-engineering: {df_engineered.shape}")

transformer = DataTransformer(df_engineered)
# Ejecutar transformaciones
print(f"\n--- Normalizando variables numéricas ---")
df_transformed = transformer.normalize_numeric_features(method='minmax')

print(f"\n--- Codificando variables categóricas ---")
#df_encoded = transformer.encode_categorical_features(method='ordinal')

print(f"\nDataset después de transformación:")
print(f"  Dimensiones: {df_transformed.shape}")
print(df_transformed.dtypes)

print(f"\nPrimeras 5 filas del dataset transformado:")
print(df_transformed.head())

print(f"\nEstadísticas descriptivas:")
print(df_transformed.describe())

print("\n" + "=" * 70)
print("✓ TODAS LAS CLASES PROBADAS EXITOSAMENTE")
print("=" * 70)

BLOQUE 3: TEST DATATRANSFORMER

Dataset antes de transformación:
  Dimensiones: (1706, 208)
idnum                   int16
pais                     int8
nationality              int8
estratopri              int16
estratosec               int8
                    ...      
sexin                    int8
colori                   int8
fecha          datetime64[ns]
formatq                  int8
idiomaq                  int8
Length: 208, dtype: object

--- Feature Engineering ---

FEATURE ENGINEERING: construcción de índices compuestos

[1/2] Índice de riqueza material (variables R):
  ✓ wealth_index creado (10 variables R → 1 índice, rango [0, 10])

[2/2] Índice de participación cívica (variables CP):
  ✓ civic_index creado (4 variables CP → 1 índice, rango [4, 16], escala invertida: mayor = más participativo)

✓ Feature engineering completado:
  Índices creados:        2
  Variables eliminadas:   14
  Dimensiones finales:    (1706, 196)
  Dimensiones post-engineering: (1706, 196)

--- Norma